In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fstreet_view_insights%2Ffull_frame%2Ffull_frame_vs_cropped_comparison.ipynb?utm_source=full_frame_street_view_insights_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
</table>

# Full-Frame vs Cropped Observation Comparison with Gemini 3.5 Flash

This notebook demonstrates how to compare detected features, input/output token counts, and API costs between Full-Frame and Cropped observation views for the same set of assets using Gemini 3.5 Flash.

## Install Required Libraries

In [ ]:
!pip install --upgrade google-cloud-bigquery google-genai google-cloud-storage "pillow<11.0.0" matplotlib

## Configuration

**Important**: Replace the placeholder values below with your actual GCP Project ID and Region.

In [ ]:
PROJECT_ID = 'imagery-insights-sandbox'  # @param {type:"string"}
REGION = 'global'      # @param {type:"string"}

# BigQuery Configuration
BIGQUERY_DATASET_ID = 'imagery_insights___us' # @param {type:"string"}
ASSET_LIMIT = 2 # @param {type:"integer"}
MODEL = "gemini-3.5-flash" # @param {type:"string"}
THINKING_LEVEL = "HIGH" # @param ["MINIMAL", "LOW", "MEDIUM", "HIGH"] {type:"string"}

## Imports and SDK Initialization

In [ ]:
import io
import vertexai
import PIL.Image
import PIL.ImageDraw
import matplotlib.pyplot as plt
from google.cloud import bigquery
from google.cloud import storage
from google import genai
from google.genai import types
from google.genai.types import Content, Part

# Initialize Vertex AI SDK and Gemini Client
vertexai.init(project=PROJECT_ID, location=REGION)
client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

## Fetch Asset IDs and Observations from BigQuery

We query BigQuery to fetch asset IDs that intersect both the Full Frame and Cropped tables, then retrieve all observations of those asset IDs from both sources.

In [ ]:
ASSET_IDS_SQL_QUERY = f"""
SELECT DISTINCT asset_id 
FROM `{PROJECT_ID}.{BIGQUERY_DATASET_ID}.full_frame_observations_latest`
INTERSECT DISTINCT
SELECT DISTINCT asset_id 
FROM `{PROJECT_ID}.{BIGQUERY_DATASET_ID}.cropped_observations_latest`
LIMIT {ASSET_LIMIT};
"""

try:
    bigquery_client = bigquery.Client(project=PROJECT_ID)
    asset_job = bigquery_client.query(ASSET_IDS_SQL_QUERY)
    asset_rows = list(asset_job)
    
    if not asset_rows:
        print("No intersecting assets found.")
        assets = []
    else:
        asset_ids = [row.get("asset_id") for row in asset_rows]
        formatted_ids = ", ".join([f"'{aid}'" for aid in asset_ids])
        
        OBSERVATIONS_SQL_QUERY = f"""
        SELECT
          'full_frame' as obs_type,
          asset_id,
          gcs_uri,
          bbox,
          asset_type
        FROM
          `{PROJECT_ID}.{BIGQUERY_DATASET_ID}.full_frame_observations_latest`
        WHERE asset_id IN ({formatted_ids})

        UNION ALL

        SELECT
          'cropped' as obs_type,
          asset_id,
          gcs_uri,
          bbox,
          asset_type
        FROM
          `{PROJECT_ID}.{BIGQUERY_DATASET_ID}.cropped_observations_latest`
        WHERE asset_id IN ({formatted_ids})
        """
        
        obs_job = bigquery_client.query(OBSERVATIONS_SQL_QUERY)
        obs_rows = list(obs_job)
        
        # Group observations by asset ID
        assets = {}
        for aid in asset_ids:
            assets[aid] = {
                "full_frame": [],
                "cropped": [],
                "asset_type": None
            }
            
        for row in obs_rows:
            aid = row.get("asset_id")
            obs_type = row.get("obs_type")
            obs_data = {
                "gcs_uri": row.get("gcs_uri"),
                "bbox": row.get("bbox")
            }
            if not assets[aid]["asset_type"]:
                assets[aid]["asset_type"] = row.get("asset_type")
            
            assets[aid][obs_type].append(obs_data)
            
        print(f"Successfully loaded {len(assets)} assets with observations.")
        for aid, data in assets.items():
            print(f"Asset: {aid} | Full Frame Obs: {len(data['full_frame'])} | Cropped Obs: {len(data['cropped'])}")
except Exception as e:
    print(f"An error occurred during data fetching: {e}")
    assets = {}

## Bounding Box Visualization Helpers

Define helpers to download images, draw bounding boxes, and display Full-Frame vs Cropped views side-by-side.

In [ ]:
def download_image(gcs_uri: str) -> PIL.Image.Image:
    parts = gcs_uri[5:].split("/", 1)
    bucket_name = parts[0]
    blob_name = parts[1]
    storage_client = storage.Client(project=PROJECT_ID)
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    image_bytes = blob.download_as_bytes()
    return PIL.Image.open(io.BytesIO(image_bytes))

def draw_bbox(image: PIL.Image.Image, bbox: dict, label: str = None) -> PIL.Image.Image:
    draw = PIL.ImageDraw.Draw(image)
    xmin = bbox['lo']['x']
    ymin = bbox['lo']['y']
    xmax = bbox['hi']['x']
    ymax = bbox['hi']['y']
    draw.rectangle([xmin, ymin, xmax, ymax], outline="red", width=10)
    if label:
        draw.text((xmin + 20, ymin + 20), label, fill="red")
    return image

def display_side_by_side(ff_obs: list, cr_obs: list):
    """
    Displays side-by-side comparison of observations.
    """
    limit = min(len(ff_obs), len(cr_obs), 3)
    print(f"Displaying first {limit} observations side-by-side...")
    for i in range(limit):
        ff_uri = ff_obs[i]["gcs_uri"]
        ff_bbox = ff_obs[i]["bbox"]
        cr_uri = cr_obs[i]["gcs_uri"]
        cr_bbox = cr_obs[i]["bbox"]
        
        try:
            ff_img = download_image(ff_uri)
            ff_img = draw_bbox(ff_img, ff_bbox, "Full Frame")
            
            cr_img = download_image(cr_uri)
            cr_img = draw_bbox(cr_img, cr_bbox, "Cropped")
            
            fig, axes = plt.subplots(1, 2, figsize=(16, 8))
            axes[0].imshow(ff_img)
            axes[0].set_title(f"Full Frame View {i+1}")
            axes[0].axis('off')
            
            axes[1].imshow(cr_img)
            axes[1].set_title(f"Cropped View {i+1}")
            axes[1].axis('off')
            
            plt.show()
        except Exception as e:
            print(f"Error displaying side-by-side for index {i}: {e}")

## Define Image Analysis Function

Define a function to send a batch of observations to Gemini for analysis and calculate the API costs.

In [ ]:
def analyze_observations(obs_list: list, prompt: str) -> tuple[str, float, int, int]:
    """
    Submits a batch of images to Gemini for unified understanding.
    Returns response text, calculated cost, prompt token count, and candidates token count.
    """
    try:
        contents = [prompt]
        for obs in obs_list:
            contents.append(Part(file_data={'file_uri': obs["gcs_uri"], 'mime_type': 'image/jpeg'}))
            
        config = types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(
                thinking_level=THINKING_LEVEL
            )
        )
        response = client.models.generate_content(model=MODEL, contents=contents, config=config)
        
        # Calculate cost dynamically from usage metadata
        prompt_tokens = response.usage_metadata.prompt_token_count
        completion_tokens = response.usage_metadata.candidates_token_count
        
        # Pricing for gemini-3.5-flash: Input: $0.000075 / 1k, Output: $0.00030 / 1k
        input_cost = prompt_tokens * (0.000075 / 1000)
        output_cost = completion_tokens * (0.00030 / 1000)
        total_cost = input_cost + output_cost
        
        return response.text, total_cost, prompt_tokens, completion_tokens
    except Exception as e:
        print(f"Error running batch analysis: {e}")
        return "Analysis failed.", 0.0, 0, 0

## Run Side-by-Side Analysis and Comparison

Loop through each of the assets, show the views side-by-side, analyze the Full-Frame batch vs the Cropped batch, and compare their predictions, tokens, and processing costs.

In [ ]:
prompt = """You will be provided with multiple photos showing different views of the exact same utility pole:
{photos_of_utility_pole}

Instructions:
1. Analyze all the provided images together. Combine the visual information to provide a single, unified report.
2. Detect and count the following across the entire asset:
    * Transformers
    * Power lines coming from the pole
    * Street lamps attached to the pole
    * Telephone or junction boxes
3. Assess the overall condition of the pole. If it appears in good condition, note \"OK\".
4. Note the material (e.g., wood, metal, concrete) the pole is made of.
5. Determine the primary type of the pole.
6. Return your findings in the following JSON format:

```json
{
  \"pole_condition\": \"OK/Damaged/Other Issues\",
  \"type\": \"<pole_type>\",
  \"material\": \"<material>\",
  \"transformers\": <number_of_transformers>,
  \"power_lines\": <number_of_power_lines>,
  \"street_lamps\": <number_of_street_lamps>,
  \"junction_boxes\": <number_of_junction_boxes>,
  \"additional_notes\": \"<summary of observations combined across all images>\"
}
```
"""

if assets:
    for aid, data in assets.items():
        ff_obs = data["full_frame"]
        cr_obs = data["cropped"]
        
        print(f"\n=========================================================================")
        print(f"ANALYZING ASSET ID: {aid}")
        print(f"=========================================================================")
        
        # 1. Visual side-by-side comparison
        display_side_by_side(ff_obs, cr_obs)
        
        # 2. Perform Full-Frame batch analysis
        print("\n--- Running Full-Frame Batch Analysis ---")
        ff_res, ff_cost, ff_in, ff_out = analyze_observations(ff_obs, prompt)
        
        # 3. Perform Cropped batch analysis
        print("\n--- Running Cropped Batch Analysis ---")
        cr_res, cr_cost, cr_in, cr_out = analyze_observations(cr_obs, prompt)
        
        # 4. Print Comparison Report
        print(f"\n--------------------------------------------")
        print(f"COMPARISON REPORT FOR {aid}")
        print(f"--------------------------------------------")
        print(f"[FULL FRAME RESULTS]:\n{ff_res}")
        print(f"Tokens: Input: {ff_in} | Output: {ff_out} | Cost: ${ff_cost:.6f}")
        print(f"\n[CROPPED RESULTS]:\n{cr_res}")
        print(f"Tokens: Input: {cr_in} | Output: {cr_out} | Cost: ${cr_cost:.6f}")
        print(f"--------------------------------------------")
else:
    print("No assets found to compare.")